In [8]:

from openai import OpenAI
import os, json, re, time, pathlib
import pandas as pd

In [9]:
client = OpenAI()

BASE_MODEL = os.getenv("BASE_MODEL", "gpt-4o-mini")   # or "gpt-4o"
FT_MODEL   = os.getenv("FT_MODEL",  "ft:gpt-3.5-turbo-0125:personal:medchefstyle-v1:COkL6OPV")

# Prompts to probe different behaviors (no legumes in some on purpose)
PROMPTS = [
    "salmon, lemon, olive oil, dill, garlic, arugula, cherry tomatoes",
    "eggplant, zucchini, bell pepper, feta cheese, basil, olive oil, tomato",
    "whole wheat pasta, mushrooms, spinach, parmesan, garlic, olive oil",
    "chicken breast, lemon, thyme, oregano, olive oil, potatoes, onion",
    "lentils, carrots, cumin, coriander, onion, tomato, parsley",        # legume allowed
    "bulgur, spinach, mint, cucumber, olive oil, lemon juice",
    "tofu, tomato, basil, olive oil, garlic, zucchini",
    "eggs, tomato, spinach, feta cheese, olive oil",
]

MODELS = {
    "Base": BASE_MODEL,
    "MedChefStyle-v1": FT_MODEL,
}

SAVE_DIR = pathlib.Path("eval_outputs")
SAVE_DIR.mkdir(exist_ok=True)

COOKING_VERBS = ["grill","bake","roast","simmer","boil","sauté","saute","fry","toast","steam","stir","sear","poach","braise"]
LEGUMES = ["chickpea","chickpeas","garbanzo","lentil","lentils","bean","beans","peas","pease"]


In [10]:
def legume_in_text(text: str) -> int:
    t = text.lower()
    return sum(1 for w in LEGUMES if w in t)

def legume_in_list(txt: str) -> bool:
    t = txt.lower()
    return any(w in t for w in LEGUMES)

def count_cooking_verbs(text: str) -> int:
    t = text.lower()
    return sum(1 for v in COOKING_VERBS if v in t)

def extract_json_or_text(s: str):
    """Try to parse assistant output as JSON with keys we expect; fallback to raw text."""
    s_stripped = s.strip()
    # Try to locate a JSON block if wrapped with text
    m = re.search(r"\{.*\}", s_stripped, flags=re.S)
    candidate = m.group(0) if m else s_stripped
    try:
        obj = json.loads(candidate)
        return obj, None
    except Exception:
        return None, s_stripped

In [11]:

def call_model(model: str, ing_list: str, temperature: float = 0.6) -> str:
    sys = (
        "You are MedChef, a Mediterranean diet recipe composer. "
        "Return STRICT JSON with keys: title, servings, cuisine, ingredients(list of {item,qty,unit,notes?}), "
        "steps(list of strings), nutrition({calories_kcal,protein_g,carbs_g,fat_g,fiber_g,sodium_mg}). "
        "Be realistic in cooking steps (soak/boil legumes, proper times). Avoid adding legumes unless explicitly present."
    )
    user = f"Compose a complete recipe using ONLY these ingredients (plus minimal pantry items if needed): { ing_list }."
    r = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role":"system","content":sys},
            {"role":"user","content":user},
        ],
    )
    return r.choices[0].message.content

In [12]:
rows = []
for prompt in PROMPTS:
    legumes_in_input = legume_in_list(prompt)
    for label, model in MODELS.items():
        print(f"→ {label} on: {prompt}")
        out = call_model(model, prompt)
        # Save raw output
        tag = re.sub(r"[^a-z0-9]+","_", prompt.lower()).strip("_")[:50]
        out_path = SAVE_DIR / f"{label}__{tag}.txt"
        out_path.write_text(out, encoding="utf-8")

        js, raw = extract_json_or_text(out)
        text_for_metrics = out.lower()

        # Prefer JSON-derived counts when possible
        if js and isinstance(js, dict):
            ingredients = js.get("ingredients") or []
            steps = js.get("steps") or []
            cals = None
            try:
                cals = js.get("nutrition",{}).get("calories_kcal", None)
            except Exception:
                cals = None
            num_ingredients = len(ingredients) if isinstance(ingredients, list) else 0
            num_steps = len(steps) if isinstance(steps, list) else 0
        else:
            num_ingredients = len(re.findall(r"ingredient", text_for_metrics))
            num_steps = len(re.findall(r"step", text_for_metrics))
            # try regex for calories if present
            mcal = re.search(r"(\d+)\s*(?:kcal|calories)", text_for_metrics)
            cals = int(mcal.group(1)) if mcal else None

        legumes_out = legume_in_text(text_for_metrics)
        cooking_verbs = count_cooking_verbs(text_for_metrics)
        legume_bias = int((not legumes_in_input) and (legumes_out > 0))

        rows.append({
            "prompt": prompt,
            "model": label,
            "legumes_in_input": int(legumes_in_input),
            "legumes_mentions_out": legumes_out,
            "legume_bias": legume_bias,          # legumes added when not in input
            "cooking_verbs_count": cooking_verbs,
            "num_ingredients_tokens": num_ingredients,
            "num_steps_tokens": num_steps,
            "calories_kcal": cals,
            "raw_file": str(out_path),
        })
        # polite pacing
        time.sleep(0.5)

→ Base on: salmon, lemon, olive oil, dill, garlic, arugula, cherry tomatoes
→ MedChefStyle-v1 on: salmon, lemon, olive oil, dill, garlic, arugula, cherry tomatoes
→ Base on: eggplant, zucchini, bell pepper, feta cheese, basil, olive oil, tomato
→ MedChefStyle-v1 on: eggplant, zucchini, bell pepper, feta cheese, basil, olive oil, tomato
→ Base on: whole wheat pasta, mushrooms, spinach, parmesan, garlic, olive oil
→ MedChefStyle-v1 on: whole wheat pasta, mushrooms, spinach, parmesan, garlic, olive oil
→ Base on: chicken breast, lemon, thyme, oregano, olive oil, potatoes, onion
→ MedChefStyle-v1 on: chicken breast, lemon, thyme, oregano, olive oil, potatoes, onion
→ Base on: lentils, carrots, cumin, coriander, onion, tomato, parsley
→ MedChefStyle-v1 on: lentils, carrots, cumin, coriander, onion, tomato, parsley
→ Base on: bulgur, spinach, mint, cucumber, olive oil, lemon juice
→ MedChefStyle-v1 on: bulgur, spinach, mint, cucumber, olive oil, lemon juice
→ Base on: tofu, tomato, basil, ol

In [13]:
df = pd.DataFrame(rows)
df.to_csv("model_comparison_results.csv", index=False)
print("✅ Saved: model_comparison_results.csv")

✅ Saved: model_comparison_results.csv


In [14]:
# Quick summary view for paper
summary = df.groupby("model").agg(
    prompts_tested=("prompt","nunique"),
    mean_legume_bias=("legume_bias","mean"),
    mean_cooking_verbs=("cooking_verbs_count","mean"),
    mean_ingredients_tokens=("num_ingredients_tokens","mean"),
    mean_steps_tokens=("num_steps_tokens","mean")
).reset_index()
print("\n=== Summary ===")
print(summary.to_string(index=False))


=== Summary ===
          model  prompts_tested  mean_legume_bias  mean_cooking_verbs  mean_ingredients_tokens  mean_steps_tokens
           Base               8               0.0               2.000                    6.875              8.875
MedChefStyle-v1               8               0.0               1.875                    6.375              5.875


In [15]:
# pip install openai pandas
# ensure: setx OPENAI_API_KEY "sk-..." before running

from openai import OpenAI
import os, json, re, time, pathlib
import pandas as pd

client = OpenAI()

BASE_MODEL = os.getenv("BASE_MODEL", "gpt-4o-mini")
FT_MODEL   = os.getenv("FT_MODEL",  "ft:gpt-3.5-turbo-0125:personal:medchefstyle-v1:COkL6OPV")

PROMPTS = [
    "salmon, lemon, olive oil, dill, garlic, arugula, cherry tomatoes",
    "eggplant, zucchini, bell pepper, feta cheese, basil, olive oil, tomato",
    "whole wheat pasta, mushrooms, spinach, parmesan, garlic, olive oil",
    "chicken breast, lemon, thyme, oregano, olive oil, potatoes, onion",
    "lentils, carrots, cumin, coriander, onion, tomato, parsley",
    "bulgur, spinach, mint, cucumber, olive oil, lemon juice",
    "tofu, tomato, basil, olive oil, garlic, zucchini",
    "eggs, tomato, spinach, feta cheese, olive oil",
]

MODELS = {
    "Base": BASE_MODEL,
    "MedChefStyle-v1": FT_MODEL,
}

SAVE_DIR = pathlib.Path("eval_outputs")
SAVE_DIR.mkdir(exist_ok=True)

COOKING_VERBS = ["grill","bake","roast","simmer","boil","sauté","saute","fry","toast","steam","stir","sear","poach","braise"]
LEGUMES = ["chickpea","chickpeas","garbanzo","lentil","lentils","bean","beans","peas"]
CULTURAL_KEYWORDS = [
    # Greek
    "spanakopita","tzatziki","souvlaki","moussaka","dolma","horiatiki",
    # Italian
    "bruschetta","caprese","pasta","lasagna","risotto","minestrone","gnocchi",
    # French
    "ratatouille","niçoise","bouillabaisse",
    # Levantine
    "tabbouleh","hummus","falafel","fattoush","shawarma",
    # Spanish
    "gazpacho","paella","tapas","tortilla española","pisto",
    # General Mediterranean
    "wrap","grilled","baked fish","olive","herbs","feta","mediterranean"
]

def legume_in_text(text: str) -> int:
    t = text.lower()
    return sum(1 for w in LEGUMES if w in t)

def legume_in_list(txt: str) -> bool:
    t = txt.lower()
    return any(w in t for w in LEGUMES)

def count_cooking_verbs(text: str) -> int:
    t = text.lower()
    return sum(1 for v in COOKING_VERBS if v in t)

def cultural_fit_score(text: str) -> int:
    """Count presence of Mediterranean archetype keywords."""
    t = text.lower()
    return sum(1 for w in CULTURAL_KEYWORDS if w in t)

def extract_json_or_text(s: str):
    """Try to parse assistant output as JSON; fallback to raw text."""
    s_stripped = s.strip()
    m = re.search(r"\{.*\}", s_stripped, flags=re.S)
    candidate = m.group(0) if m else s_stripped
    try:
        obj = json.loads(candidate)
        return obj, None
    except Exception:
        return None, s_stripped

def call_model(model: str, ing_list: str, temperature: float = 0.6) -> str:
    sys = (
        "You are MedChef, a Mediterranean diet recipe composer. "
        "Return STRICT JSON with keys: title, ingredients(list), steps(list), nutrition(dict). "
        "Avoid adding legumes unless explicitly listed. Use authentic Mediterranean names if applicable."
    )
    user = f"Compose a complete Mediterranean-style recipe using ONLY these ingredients (plus minimal pantry staples): {ing_list}."
    r = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": sys},
            {"role": "user", "content": user},
        ],
    )
    return r.choices[0].message.content

rows = []
for prompt in PROMPTS:
    legumes_in_input = legume_in_list(prompt)
    for label, model in MODELS.items():
        print(f"→ {label}: {prompt}")
        out = call_model(model, prompt)
        # Save raw output
        tag = re.sub(r"[^a-z0-9]+","_", prompt.lower()).strip("_")[:50]
        out_path = SAVE_DIR / f"{label}__{tag}.txt"
        out_path.write_text(out, encoding="utf-8")

        js, raw = extract_json_or_text(out)
        text_for_metrics = out.lower()

        # Prefer JSON-derived counts when possible
        if js and isinstance(js, dict):
            ingredients = js.get("ingredients") or []
            steps = js.get("steps") or []
            num_ingredients = len(ingredients) if isinstance(ingredients, list) else 0
            num_steps = len(steps) if isinstance(steps, list) else 0
        else:
            num_ingredients = len(re.findall(r"ingredient", text_for_metrics))
            num_steps = len(re.findall(r"step", text_for_metrics))

        legumes_out = legume_in_text(text_for_metrics)
        cooking_verbs = count_cooking_verbs(text_for_metrics)
        cultural_score = cultural_fit_score(text_for_metrics)
        legume_bias = int((not legumes_in_input) and (legumes_out > 0))

        rows.append({
            "prompt": prompt,
            "model": label,
            "legumes_in_input": int(legumes_in_input),
            "legumes_mentions_out": legumes_out,
            "legume_bias": legume_bias,
            "cooking_verbs_count": cooking_verbs,
            "num_ingredients_tokens": num_ingredients,
            "num_steps_tokens": num_steps,
            "cultural_fit_score": cultural_score,
            "raw_file": str(out_path),
        })
        time.sleep(0.4)

df = pd.DataFrame(rows)
df.to_csv("model_comparison_with_culturalfit.csv", index=False)
print("✅ Saved: model_comparison_with_culturalfit.csv")

# ---- Summary ----
summary = df.groupby("model").agg(
    prompts_tested=("prompt","nunique"),
    mean_legume_bias=("legume_bias","mean"),
    mean_cooking_verbs=("cooking_verbs_count","mean"),
    mean_cultural_fit=("cultural_fit_score","mean"),
    mean_ingredients_tokens=("num_ingredients_tokens","mean"),
    mean_steps_tokens=("num_steps_tokens","mean")
).reset_index()

print("\n=== Summary ===")
print(summary.to_string(index=False))

→ Base: salmon, lemon, olive oil, dill, garlic, arugula, cherry tomatoes
→ MedChefStyle-v1: salmon, lemon, olive oil, dill, garlic, arugula, cherry tomatoes
→ Base: eggplant, zucchini, bell pepper, feta cheese, basil, olive oil, tomato
→ MedChefStyle-v1: eggplant, zucchini, bell pepper, feta cheese, basil, olive oil, tomato
→ Base: whole wheat pasta, mushrooms, spinach, parmesan, garlic, olive oil
→ MedChefStyle-v1: whole wheat pasta, mushrooms, spinach, parmesan, garlic, olive oil
→ Base: chicken breast, lemon, thyme, oregano, olive oil, potatoes, onion
→ MedChefStyle-v1: chicken breast, lemon, thyme, oregano, olive oil, potatoes, onion
→ Base: lentils, carrots, cumin, coriander, onion, tomato, parsley
→ MedChefStyle-v1: lentils, carrots, cumin, coriander, onion, tomato, parsley
→ Base: bulgur, spinach, mint, cucumber, olive oil, lemon juice
→ MedChefStyle-v1: bulgur, spinach, mint, cucumber, olive oil, lemon juice
→ Base: tofu, tomato, basil, olive oil, garlic, zucchini
→ MedChefStyl